In [5]:
# =========================================================
# FRAMEWORK MANUAL — BASE ESTRUTURAL
# IMDb + Hubara como primeiro caso
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
import pandas as pd


# ---------------------------------------------------------
# 1) Especificação do cenário
# ---------------------------------------------------------
# Representa o "mundo" que será testado:
# - nome do dataset
# - entidades
# - relacionamentos
# - queries do workload
#
# Exemplo:
# dataset = IMDb
# entities = User, Movie, Rating, ...
@dataclass
class PhysicalDBSpec:
    model: str
    engine: str
    host: str = "localhost"
    port: Optional[int] = None
    database_name: Optional[str] = None
    username: Optional[str] = None
    password: Optional[str] = None

    # opcionais, úteis para bancos como MongoDB
    connection_uri: Optional[str] = None
    options: Dict[str, Any] = field(default_factory=dict)

# ---------------------------------------------------------
# 2) Fragmento recomendado
# ---------------------------------------------------------
# Cada fragmento é um grupo de entidades que será alocado
# em um tipo/modelo de banco.
#
# Exemplo:
# Fragmento 1 = ['User'] -> relational
# Fragmento 2 = ['Movie', 'Genre', ...] -> column
@dataclass
class FragmentSpec:
    name: str
    entities: List[str]
    model: str   # relational, document, column, graph, etc.
    notes: str = ""


# ---------------------------------------------------------
# 3) Recomendação final
# ---------------------------------------------------------
# Representa a saída final de um método como o Hubara.
#
# Exemplo:
# recommendation_name = "Hubara IMDb"
# fragments = [FragmentSpec(...), FragmentSpec(...)]
@dataclass
class RecommendationSpec:
    recommendation_name: str
    source_method: str
    source_dataset: str
    fragments: List[FragmentSpec]


# ---------------------------------------------------------
# 4) Banco físico concreto
# ---------------------------------------------------------
# Aqui diferenciamos "modelo" de "produto".
#
# Exemplo:
# model = relational
# engine = PostgreSQL
#
# ou:
# model = document
# engine = MongoDB
@dataclass
class PhysicalDBSpec:
    model: str
    engine: str
    host: str = "localhost"
    port: Optional[int] = None
    database_name: Optional[str] = None
    username: Optional[str] = None
    password: Optional[str] = None


# ---------------------------------------------------------
# 5) Plano de materialização
# ---------------------------------------------------------
# Traduz a recomendação abstrata para bancos concretos.
#
# Exemplo:
# fragment User -> PostgreSQL
# fragment Content -> Cassandra
@dataclass
class MaterializationPlan:
    scenario_name: str
    recommendation_name: str
    fragment_to_db: Dict[str, PhysicalDBSpec]


# ---------------------------------------------------------
# 6) Especificação de uma query do benchmark
# ---------------------------------------------------------
# Aqui vamos guardar:
# - nome da query
# - tipo
# - entidades tocadas
# - texto abstrato
#
# Mais tarde, podemos ter versões por banco:
# sql_text, mongo_text, cypher_text, etc.
@dataclass
class BenchmarkQuery:
    name: str
    query_type: str
    entities_involved: List[str]
    abstract_query: str
    expected_fragment: Optional[str] = None
    expected_db_model: Optional[str] = None


# ---------------------------------------------------------
# 7) Especificação do workload
# ---------------------------------------------------------
@dataclass
class WorkloadSpec:
    name: str
    description: str
    queries: List[BenchmarkQuery]
    repetitions: int = 10


# ---------------------------------------------------------
# 8) Resultado de execução de uma query
# ---------------------------------------------------------
@dataclass
class QueryExecutionResult:
    query_name: str
    fragment_name: str
    db_engine: str
    run_id: int
    benchmark_phase: str   # "cold" ou "hot"
    latency_ms: float
    success: bool
    error_message: Optional[str] = None

# ---------------------------------------------------------
# 9) Resultado consolidado do benchmark
# ---------------------------------------------------------
@dataclass
class BenchmarkResult:
    scenario_name: str
    recommendation_name: str
    query_results: List[QueryExecutionResult] = field(default_factory=list)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "scenario_name": self.scenario_name,
                "recommendation_name": self.recommendation_name,
                "query_name": r.query_name,
                "fragment_name": r.fragment_name,
                "db_engine": r.db_engine,
                "run_id": r.run_id,
                "benchmark_phase": r.benchmark_phase,
                "latency_ms": r.latency_ms,
                "success": r.success,
                "error_message": r.error_message,
            }
            for r in self.query_results
        ])

In [32]:
# =========================================================
# INSTÂNCIA 1 — CENÁRIO IMDb + RECOMENDAÇÃO HUBARA
# =========================================================

# ---------------------------------------------------------
# 1) Cenário IMDb
# ---------------------------------------------------------
imdb_scenario = ScenarioSpec(
    name="IMDb",
    description="IMDb-like scenario based on the Hubara paper",
    entities=[
        "Person",
        "User",
        "Genre",
        "Role",
        "Rate",
        "WatchItem",
        "Series",
        "Movie",
        "Episode",
    ],
    relationships=[
        {"source": "Person", "target": "Role", "name": "acts_in_via_role"},
        {"source": "Role", "target": "WatchItem", "name": "role_of_watchitem"},
        {"source": "Person", "target": "WatchItem", "name": "directs"},
        {"source": "Person", "target": "WatchItem", "name": "produces"},
        {"source": "Genre", "target": "WatchItem", "name": "has_genre"},
        {"source": "User", "target": "Rate", "name": "rates"},
        {"source": "Rate", "target": "WatchItem", "name": "rate_of_watchitem"},
        {"source": "Series", "target": "WatchItem", "name": "series_is_watchitem"},
        {"source": "Movie", "target": "WatchItem", "name": "movie_is_watchitem"},
        {"source": "Episode", "target": "WatchItem", "name": "episode_is_watchitem"},
        {"source": "Series", "target": "Episode", "name": "series_contains_episode"},
    ],
    workload_queries=[
        {
            "name": "Q1_Login",
            "type": "select",
            "entities": ["User"],
            "abstract_query": "RETURN password FROM User WHERE username = ?"
        },
        {
            "name": "Q2_SimpleSearch",
            "type": "select",
            "entities": ["WatchItem"],
            "abstract_query": "RETURN ALL FROM WatchItem WHERE title = ?"
        },
        {
            "name": "Q3_AddEntitiesAndRelationships",
            "type": "insert",
            "entities": ["Person", "WatchItem", "Role"],
            "abstract_query": "INSERT Person and connect Person with WatchItem"
        },
        {
            "name": "Q4_Recommendation",
            "type": "select",
            "entities": ["WatchItem", "Genre", "Movie", "Series", "Episode"],
            "abstract_query": "Recommendation query by genre and type"
        },
        {
            "name": "Q5_AllPersonsOfTypeForWatchItem",
            "type": "select",
            "entities": ["WatchItem", "Person", "Role"],
            "abstract_query": "RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor"
        },
    ]
)


# ---------------------------------------------------------
# 2) Recomendação Hubara para IMDb
# ---------------------------------------------------------
# Resultado que você reproduziu:
# - Fragmento de conteúdo -> Column
# - Fragmento de User -> RDBMS
hubara_imdb_recommendation = RecommendationSpec(
    recommendation_name="Hubara_IMDb_Recommendation",
    source_method="Hubara",
    source_dataset="IMDb",
    fragments=[
        FragmentSpec(
            name="ContentFragment",
            entities=["Episode", "Genre", "Movie", "Person", "Rate", "Role", "Series", "WatchItem"],
            model="column",
            notes="Recommended by the implemented Hubara workflow"
        ),
        FragmentSpec(
            name="UserFragment",
            entities=["User"],
            model="relational",
            notes="Recommended by the implemented Hubara workflow"
        ),
    ]
)


# ---------------------------------------------------------
# 3) Plano físico concreto
# ---------------------------------------------------------
# Aqui você escolhe quais produtos concretos representam
# cada modelo lógico.
#
# Exemplo inicial:
# relational -> PostgreSQL
# column -> Cassandra
#
# Você pode trocar Cassandra por outro depois.
hubara_imdb_materialization = MaterializationPlan(
    scenario_name="IMDb",
    recommendation_name="Hubara_IMDb_Recommendation",
    fragment_to_db={
        "UserFragment": PhysicalDBSpec(
            model="relational",
            engine="PostgreSQL",
            host="127.0.0.1",
            port=55432,
            database_name="imdb_user_db",
            username="postgres",
            password="postgres"
        ),
        "ContentFragment": PhysicalDBSpec(
            model="column",
            engine="Cassandra",
            host="127.0.0.1",
            port=59042,
            database_name="imdb_content_db"
        ),
    }
)


# =========================================================
# BLOCO M1 — MATERIALIZAÇÃO ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

hubara_imdb_mongo_materialization = MaterializationPlan(
    scenario_name="IMDb",
    recommendation_name="Hubara_IMDb_Recommendation_MongoAlternative",
    fragment_to_db={
        "UserFragment": PhysicalDBSpec(
            model="relational",
            engine="PostgreSQL",
            host="127.0.0.1",
            port=55432,   # via túnel SSH, como você já configurou
            database_name="imdb_user_db_mongo_alt",
            username="postgres",
            password="postgres"
        ),
        "ContentFragment": PhysicalDBSpec(
            model="document",
            engine="MongoDB",
            host="127.0.0.1",
            port=57017,
            database_name="imdb_content_mongo_db",
            username="mongo",
            password="mongo"
        ),
    }
)

print("Materialização alternativa criada com sucesso.")
print(hubara_imdb_mongo_materialization)


# ---------------------------------------------------------
# 4) Workload do benchmark
# ---------------------------------------------------------
imdb_workload = WorkloadSpec(
    name="IMDb_Hubara_Workload",
    description="Benchmark workload for the IMDb scenario using the Hubara recommendation",
    queries=[
        BenchmarkQuery(
            name="Q1_Login",
            query_type="select",
            entities_involved=["User"],
            abstract_query="RETURN password FROM User WHERE username = ?",
            expected_fragment="UserFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q2_SimpleSearch",
            query_type="select",
            entities_involved=["WatchItem"],
            abstract_query="RETURN ALL FROM WatchItem WHERE title = ?",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
        BenchmarkQuery(
            name="Q3_AddEntitiesAndRelationships",
            query_type="insert",
            entities_involved=["Person", "WatchItem", "Role"],
            abstract_query="INSERT Person and connect Person with WatchItem",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
        BenchmarkQuery(
            name="Q4_Recommendation",
            query_type="select",
            entities_involved=["WatchItem", "Genre", "Movie", "Series", "Episode"],
            abstract_query="Recommendation query by genre and type",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
        BenchmarkQuery(
            name="Q5_AllPersonsOfTypeForWatchItem",
            query_type="select",
            entities_involved=["WatchItem", "Person", "Role"],
            abstract_query="RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor",
            expected_fragment="ContentFragment",
            expected_db_model="column"
        ),
    ],
    repetitions=10
)


# ---------------------------------------------------------
# 4) Workload do benchmark — alternativa MongoDB
# ---------------------------------------------------------
imdb_workload_mongo = WorkloadSpec(
    name="IMDb_Hubara_Workload_Mongo",
    description="Benchmark workload for the IMDb scenario using the Hubara recommendation with MongoDB for the content fragment",
    queries=[
        BenchmarkQuery(
            name="Q1_Login",
            query_type="select",
            entities_involved=["User"],
            abstract_query="RETURN password FROM User WHERE username = ?",
            expected_fragment="UserFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q2_SimpleSearch",
            query_type="select",
            entities_involved=["WatchItem"],
            abstract_query="RETURN ALL FROM WatchItem WHERE title = ?",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="Q3_AddEntitiesAndRelationships",
            query_type="insert",
            entities_involved=["Person", "WatchItem", "Role"],
            abstract_query="INSERT Person and connect Person with WatchItem",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="Q4_Recommendation",
            query_type="select",
            entities_involved=["WatchItem", "Genre", "Movie", "Series", "Episode"],
            abstract_query="Recommendation query by genre and type",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
        BenchmarkQuery(
            name="Q5_AllPersonsOfTypeForWatchItem",
            query_type="select",
            entities_involved=["WatchItem", "Person", "Role"],
            abstract_query="RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor",
            expected_fragment="ContentFragment",
            expected_db_model="document"
        ),
    ],
    repetitions=10
)

Materialização alternativa criada com sucesso.
MaterializationPlan(scenario_name='IMDb', recommendation_name='Hubara_IMDb_Recommendation_MongoAlternative', fragment_to_db={'UserFragment': PhysicalDBSpec(model='relational', engine='PostgreSQL', host='127.0.0.1', port=55432, database_name='imdb_user_db_mongo_alt', username='postgres', password='postgres'), 'ContentFragment': PhysicalDBSpec(model='document', engine='MongoDB', host='127.0.0.1', port=57017, database_name='imdb_content_mongo_db', username='mongo', password='mongo')})


In [7]:
# =========================================================
# VISUALIZAÇÃO DAS ESTRUTURAS — ALTERNATIVA MONGODB
# =========================================================

# ---------------------------------------------------------
# 1) Entidades do cenário
# ---------------------------------------------------------
scenario_entities_df = pd.DataFrame({"entity": imdb_scenario.entities})

# ---------------------------------------------------------
# 2) Relacionamentos do cenário
# ---------------------------------------------------------
scenario_relationships_df = pd.DataFrame(imdb_scenario.relationships)

# ---------------------------------------------------------
# 3) Recommendation view
# Aqui usamos:
# - as entidades dos fragmentos vindas da recomendação lógica
# - o modelo vindo da materialização alternativa Mongo
# ---------------------------------------------------------
fragment_entities_map = {
    frag.name: frag.entities
    for frag in hubara_imdb_recommendation.fragments
}

recommendation_df = pd.DataFrame([
    {
        "fragment": fragment_name,
        "entities": fragment_entities_map.get(fragment_name, []),
        "model": dbspec.model,
        "engine": dbspec.engine,
        "notes": f"Alternative physical configuration using {dbspec.engine}"
    }
    for fragment_name, dbspec in hubara_imdb_mongo_materialization.fragment_to_db.items()
])

# ---------------------------------------------------------
# 4) Materialization plan
# ---------------------------------------------------------
materialization_df = pd.DataFrame([
    {
        "fragment": fragment_name,
        "model": dbspec.model,
        "engine": dbspec.engine,
        "host": dbspec.host,
        "port": dbspec.port,
        "database_name": dbspec.database_name
    }
    for fragment_name, dbspec in hubara_imdb_mongo_materialization.fragment_to_db.items()
])

# ---------------------------------------------------------
# 5) Workload view
# Aqui usamos o workload específico da alternativa Mongo
# ---------------------------------------------------------
workload_df = pd.DataFrame([
    {
        "query_name": q.name,
        "query_type": q.query_type,
        "entities_involved": q.entities_involved,
        "expected_fragment": q.expected_fragment,
        "expected_db_model": q.expected_db_model,
        "abstract_query": q.abstract_query
    }
    for q in imdb_workload_mongo.queries
])

# ---------------------------------------------------------
# 6) Exibição
# ---------------------------------------------------------
print("Entities")
display(scenario_entities_df)

print("Relationships")
display(scenario_relationships_df)

print("Recommendation")
display(recommendation_df)

print("Materialization plan")
display(materialization_df)

print("Workload")
display(workload_df)

Entities


,entity
0,Person
1,User
2,Genre
3,Role
4,Rate
5,WatchItem
6,Series
7,Movie
8,Episode


Relationships


,source,target,name
0,Person,Role,acts_in_via_role
1,Role,WatchItem,role_of_watchitem
2,Person,WatchItem,directs
3,Person,WatchItem,produces
4,Genre,WatchItem,has_genre
5,User,Rate,rates
6,Rate,WatchItem,rate_of_watchitem
7,Series,WatchItem,series_is_watchitem
8,Movie,WatchItem,movie_is_watchitem
9,Episode,WatchItem,episode_is_watchitem


Recommendation


,fragment,entities,model,engine,notes
0,UserFragment,[User],relational,PostgreSQL,Alternative physical configuration using Postg...
1,ContentFragment,"[Episode, Genre, Movie, Person, Rate, Role, Se...",document,MongoDB,Alternative physical configuration using MongoDB


Materialization plan


,fragment,model,engine,host,port,database_name
0,UserFragment,relational,PostgreSQL,127.0.0.1,55432,imdb_user_db_mongo_alt
1,ContentFragment,document,MongoDB,127.0.0.1,27017,imdb_content_mongo_db


Workload


,query_name,query_type,entities_involved,expected_fragment,expected_db_model,abstract_query
0,Q1_Login,select,[User],UserFragment,relational,RETURN password FROM User WHERE username = ?
1,Q2_SimpleSearch,select,[WatchItem],ContentFragment,document,RETURN ALL FROM WatchItem WHERE title = ?
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",ContentFragment,document,INSERT Person and connect Person with WatchItem
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",ContentFragment,document,Recommendation query by genre and type
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]",ContentFragment,document,"RETURN Person.ALL FROM WatchItem, Person WHERE..."


In [8]:
# =========================================================
# BLOCO 1 — GERADOR DE DADOS SINTÉTICOS IMDb
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List
import pandas as pd

@dataclass
class ScenarioDataBundle:
    dataset_name: str
    tables: Dict[str, pd.DataFrame] = field(default_factory=dict)

    def table_names(self) -> List[str]:
        return list(self.tables.keys())

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "table_name": name,
                "rows": len(df),
                "columns": list(df.columns)
            }
            for name, df in self.tables.items()
        ]).sort_values("table_name").reset_index(drop=True)

In [9]:
# =========================================================
# BLOCO 2 — GERADOR DE DADOS SINTÉTICOS IMDb (CORRIGIDO)
# =========================================================

import random
import numpy as np
import pandas as pd

def generate_imdb_synthetic_data(
    n_users: int = 100,
    n_persons: int = 120,
    n_watchitems: int = 80,
    n_genres: int = 10,
    seed: int = 42
) -> "ScenarioDataBundle":
    """
    Gera um dataset sintético simplificado para o cenário IMDb.
    """

    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    # 1) USERS
    users = pd.DataFrame([
        {
            "user_id": i,
            "username": f"user_{i}",
            "password": f"pass_{i}",
            "email": f"user_{i}@mail.com",
            "last_login": f"2026-01-{(i % 28) + 1:02d}"
        }
        for i in range(1, n_users + 1)
    ])

    # 2) PERSONS
    persons = pd.DataFrame([
        {
            "person_id": i,
            "name": f"Person {i}",
            "date_of_birth": f"{1970 + (i % 30)}-{(i % 12) + 1:02d}-{(i % 28) + 1:02d}",
            "gender": "M" if i % 2 == 0 else "F"
        }
        for i in range(1, n_persons + 1)
    ])

    # 3) GENRES
    genre_names = [
        "Action", "Drama", "Comedy", "Thriller", "SciFi",
        "Fantasy", "Romance", "Crime", "Adventure", "Mystery"
    ][:n_genres]

    genres = pd.DataFrame([
        {"genre_id": i + 1, "name": genre_names[i]}
        for i in range(len(genre_names))
    ])

    # 4) WATCHITEMS
    item_types = np_rng.choice(
        ["Movie", "Series", "Episode"],
        size=n_watchitems,
        p=[0.45, 0.20, 0.35]
    )

    watchitems_rows = []
    for i in range(1, n_watchitems + 1):
        genre_id = int(np_rng.integers(1, len(genres) + 1))
        release_year = int(np_rng.integers(1990, 2026))

        watchitems_rows.append({
            "watchitem_id": i,
            "title": f"Title {i}",
            "release_year": release_year,
            "avg_rating": round(float(np_rng.uniform(1.0, 10.0)), 2),
            "genre_id": genre_id,
            "item_type": item_types[i - 1]
        })

    watchitems = pd.DataFrame(watchitems_rows)

    # 5) MOVIES
    movies = watchitems[watchitems["item_type"] == "Movie"][["watchitem_id"]].copy()
    movies["length_min"] = np_rng.integers(80, 181, size=len(movies))
    movies["media"] = np_rng.choice(["Cinema", "Streaming", "TV"], size=len(movies))
    movies["income"] = np_rng.integers(100000, 1000000000, size=len(movies))

    # 6) SERIES
    series = watchitems[watchitems["item_type"] == "Series"][["watchitem_id"]].copy()
    series["seasons"] = np_rng.integers(1, 8, size=len(series))
    series["network"] = np_rng.choice(["HBO", "Netflix", "Prime", "Disney"], size=len(series))

    # 7) EPISODES
    episodes = watchitems[watchitems["item_type"] == "Episode"][["watchitem_id"]].copy()
    episodes["season"] = np_rng.integers(1, 8, size=len(episodes))
    episodes["episode_number"] = np_rng.integers(1, 25, size=len(episodes))
    episodes["length_min"] = np_rng.integers(20, 61, size=len(episodes))

    if len(series) > 0 and len(episodes) > 0:
        series_ids = series["watchitem_id"].tolist()
        episodes["series_watchitem_id"] = [rng.choice(series_ids) for _ in range(len(episodes))]
    else:
        episodes["series_watchitem_id"] = None

    # 8) ROLES
    role_types = ["Actor", "Director", "Producer"]

    roles_rows = []
    role_id = 1
    for watchitem_id in watchitems["watchitem_id"]:
        n_links = int(np_rng.integers(2, 6))
        chosen_persons = rng.sample(persons["person_id"].tolist(), k=min(n_links, len(persons)))

        for person_id in chosen_persons:
            roles_rows.append({
                "role_id": role_id,
                "person_id": person_id,
                "watchitem_id": watchitem_id,
                "role_type": rng.choice(role_types)
            })
            role_id += 1

    roles = pd.DataFrame(roles_rows)

    # 9) RATES
    rates_rows = []
    rate_id = 1

    for user_id in users["user_id"]:
        n_user_ratings = int(np_rng.integers(2, 10))
        chosen_items = rng.sample(
            watchitems["watchitem_id"].tolist(),
            k=min(n_user_ratings, len(watchitems))
        )

        for watchitem_id in chosen_items:
            rating_value = int(np_rng.integers(1, 11))
            verbal = (
                "Excellent" if rating_value >= 9 else
                "Good" if rating_value >= 7 else
                "Average" if rating_value >= 5 else
                "Bad"
            )

            rates_rows.append({
                "rate_id": rate_id,
                "user_id": user_id,
                "watchitem_id": watchitem_id,
                "rating": rating_value,
                "verbal_rating": verbal
            })
            rate_id += 1

    rates = pd.DataFrame(rates_rows)

    return ScenarioDataBundle(
        dataset_name="IMDb",
        tables={
            "users": users,
            "persons": persons,
            "genres": genres,
            "watchitems": watchitems,
            "movies": movies,
            "series": series,
            "episodes": episodes,
            "roles": roles,
            "rates": rates,
        }
    )

In [10]:
# =========================================================
# BLOCO 3 — GERAR O DATASET IMDb SINTÉTICO
# =========================================================

import pandas as pd

# Checagem simples para evitar erro confuso se o bloco anterior não tiver rodado
if "generate_imdb_synthetic_data" not in globals():
    raise NameError(
        "A função 'generate_imdb_synthetic_data' não está definida. "
        "Rode primeiro o bloco anterior que cria essa função."
    )

# Gera o dataset sintético
imdb_data_bundle = generate_imdb_synthetic_data(
    n_users=100,
    n_persons=120,
    n_watchitems=80,
    n_genres=10,
    seed=42
)

# Mostra um resumo das tabelas geradas
print("Resumo das tabelas geradas:")
display(imdb_data_bundle.summary())

# Mostra algumas amostras para inspeção
print("Prévia: users")
display(imdb_data_bundle.tables["users"].head())

print("Prévia: watchitems")
display(imdb_data_bundle.tables["watchitems"].head())

print("Prévia: roles")
display(imdb_data_bundle.tables["roles"].head())

print("Prévia: rates")
display(imdb_data_bundle.tables["rates"].head())

Resumo das tabelas geradas:


,table_name,rows,columns
0,episodes,31,"[watchitem_id, season, episode_number, length_..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[watchitem_id, length_min, media, income]"
3,persons,120,"[person_id, name, date_of_birth, gender]"
4,rates,547,"[rate_id, user_id, watchitem_id, rating, verba..."
5,roles,290,"[role_id, person_id, watchitem_id, role_type]"
6,series,15,"[watchitem_id, seasons, network]"
7,users,100,"[user_id, username, password, email, last_login]"
8,watchitems,80,"[watchitem_id, title, release_year, avg_rating..."


Prévia: users


,user_id,username,password,email,last_login
0,1,user_1,pass_1,user_1@mail.com,2026-01-02
1,2,user_2,pass_2,user_2@mail.com,2026-01-03
2,3,user_3,pass_3,user_3@mail.com,2026-01-04
3,4,user_4,pass_4,user_4@mail.com,2026-01-05
4,5,user_5,pass_5,user_5@mail.com,2026-01-06


Prévia: watchitems


,watchitem_id,title,release_year,avg_rating,genre_id,item_type
0,1,Title 1,2013,4.66,10,Episode
1,2,Title 2,2019,2.50,5,Movie
2,3,Title 3,1990,1.81,4,Episode
3,4,Title 4,2016,5.16,8,Episode
4,5,Title 5,1995,5.51,8,Movie


Prévia: roles


,role_id,person_id,watchitem_id,role_type
0,1,54,1,Producer
1,2,29,1,Director
2,3,58,1,Actor
3,4,98,2,Producer
4,5,104,2,Director


Prévia: rates


,rate_id,user_id,watchitem_id,rating,verbal_rating
0,1,1,26,1,Bad
1,2,1,47,1,Bad
2,3,1,56,5,Average
3,4,1,9,8,Good
4,5,1,43,9,Excellent


In [11]:
# =========================================================
# BLOCO 4 — PARTICIONAR OS DADOS PELOS FRAGMENTOS
# =========================================================

# Mapa simples: qual tabela lógica pertence a qual entidade conceitual
LOGICAL_TABLE_TO_ENTITY = {
    "users": "User",
    "persons": "Person",
    "genres": "Genre",
    "watchitems": "WatchItem",
    "movies": "Movie",
    "series": "Series",
    "episodes": "Episode",
    "roles": "Role",
    "rates": "Rate",
}


def split_bundle_by_recommendation(
    bundle: ScenarioDataBundle,
    recommendation: RecommendationSpec
) -> Dict[str, Dict[str, pd.DataFrame]]:
    """
    Retorna:
    {
        "UserFragment": {
            "users": df
        },
        "ContentFragment": {
            "persons": df,
            "genres": df,
            ...
        }
    }
    """

    fragment_map = {frag.name: set(frag.entities) for frag in recommendation.fragments}
    result = {frag.name: {} for frag in recommendation.fragments}

    for table_name, df in bundle.tables.items():
        entity_name = LOGICAL_TABLE_TO_ENTITY[table_name]

        for fragment_name, fragment_entities in fragment_map.items():
            if entity_name in fragment_entities:
                result[fragment_name][table_name] = df.copy()
                break

    return result

In [19]:
# =========================================================
# BLOCO 5 — EXECUTAR PARTICIONAMENTO DOS DADOS
# =========================================================

fragmented_data = split_bundle_by_recommendation(
    bundle=imdb_data_bundle,
    recommendation=hubara_imdb_recommendation
)

for fragment_name, tables_dict in fragmented_data.items():
    print("\n" + "=" * 70)
    print(f"Fragmento: {fragment_name}")

    summary_df = pd.DataFrame([
        {
            "table_name": table_name,
            "rows": len(df),
            "columns": list(df.columns)
        }
        for table_name, df in tables_dict.items()
    ]).sort_values("table_name").reset_index(drop=True)

    display(summary_df)


Fragmento: ContentFragment


,table_name,rows,columns
0,episodes,31,"[watchitem_id, season, episode_number, length_..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[watchitem_id, length_min, media, income]"
3,persons,120,"[person_id, name, date_of_birth, gender]"
4,rates,547,"[rate_id, user_id, watchitem_id, rating, verba..."
5,roles,290,"[role_id, person_id, watchitem_id, role_type]"
6,series,15,"[watchitem_id, seasons, network]"
7,watchitems,80,"[watchitem_id, title, release_year, avg_rating..."



Fragmento: UserFragment


,table_name,rows,columns
0,users,100,"[user_id, username, password, email, last_login]"


In [20]:
# =========================================================
# BLOCO 6 — PREPARAÇÃO DE PAYLOAD FÍSICO INICIAL
# =========================================================

@dataclass
class PhysicalLoadBundle:
    fragment_name: str
    db_model: str
    db_engine: str
    physical_tables: Dict[str, pd.DataFrame] = field(default_factory=dict)

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "physical_table": name,
                "rows": len(df),
                "columns": list(df.columns)
            }
            for name, df in self.physical_tables.items()
        ]).sort_values("physical_table").reset_index(drop=True)


def build_user_fragment_postgres_payload(fragment_tables: Dict[str, pd.DataFrame]) -> PhysicalLoadBundle:
    """
    Materialização simples do fragmento User para PostgreSQL.
    """
    users = fragment_tables["users"].copy()

    return PhysicalLoadBundle(
        fragment_name="UserFragment",
        db_model="relational",
        db_engine="PostgreSQL",
        physical_tables={
            "users": users
        }
    )

"""
def build_content_fragment_cassandra_payload(fragment_tables: Dict[str, pd.DataFrame]) -> PhysicalLoadBundle:
  

    persons = fragment_tables["persons"].copy()
    genres = fragment_tables["genres"].copy()
    watchitems = fragment_tables["watchitems"].copy()
    movies = fragment_tables["movies"].copy()
    series = fragment_tables["series"].copy()
    episodes = fragment_tables["episodes"].copy()
    roles = fragment_tables["roles"].copy()
    rates = fragment_tables["rates"].copy()

    # -----------------------------------------------------
    # 1) Tabela canônica de watchitems por título
    # Útil para Q2: simple search por title
    # -----------------------------------------------------
    watchitems_by_title = watchitems.copy()

    # -----------------------------------------------------
    # 2) Pessoas por watchitem
    # Útil para Q5: pessoas associadas a um item
    # -----------------------------------------------------
    persons_by_watchitem = (
        roles.merge(persons, on="person_id", how="left")
        [["watchitem_id", "person_id", "name", "role_type"]]
        .sort_values(["watchitem_id", "role_type", "person_id"])
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # 3) Watchitems por gênero e tipo
    # Útil para Q4: recommendation query
    # -----------------------------------------------------
    watchitems_by_genre_type = (
        watchitems.merge(genres, on="genre_id", how="left")
        [["genre_id", "name", "item_type", "watchitem_id", "title", "release_year", "avg_rating"]]
        .rename(columns={"name": "genre_name"})
        .sort_values(["genre_id", "item_type", "release_year"])
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # 4) Tabelas canônicas adicionais
    # -----------------------------------------------------
    return PhysicalLoadBundle(
        fragment_name="ContentFragment",
        db_model="column",
        db_engine="Cassandra",
        physical_tables={
            "persons": persons,
            "genres": genres,
            "watchitems": watchitems,
            "movies": movies,
            "series": series,
            "episodes": episodes,
            "roles": roles,
            "rates": rates,
            "watchitems_by_title": watchitems_by_title,
            "persons_by_watchitem": persons_by_watchitem,
            "watchitems_by_genre_type": watchitems_by_genre_type,
        }
    )

"""
# =========================================================
# BLOCO M2 — BUILD DO PAYLOAD FÍSICO PARA MONGODB
# =========================================================

def build_content_fragment_mongodb_payload(fragment_tables: Dict[str, pd.DataFrame]) -> PhysicalLoadBundle:
    """
    Materialização inicial do fragmento de conteúdo para MongoDB.

    Estratégia:
    - manter coleções canônicas;
    - criar coleções orientadas ao workload para facilitar comparações
      com a versão em Cassandra.
    """

    persons = fragment_tables["persons"].copy()
    genres = fragment_tables["genres"].copy()
    watchitems = fragment_tables["watchitems"].copy()
    movies = fragment_tables["movies"].copy()
    series = fragment_tables["series"].copy()
    episodes = fragment_tables["episodes"].copy()
    roles = fragment_tables["roles"].copy()
    rates = fragment_tables["rates"].copy()

    # -----------------------------------------------------
    # 1) Coleção para busca por título (Q2)
    # -----------------------------------------------------
    watchitems_by_title = watchitems.copy()

    # -----------------------------------------------------
    # 2) Coleção para pessoas por watchitem (Q5)
    # -----------------------------------------------------
    persons_by_watchitem = (
        roles.merge(persons, on="person_id", how="left")
        [["watchitem_id", "person_id", "name", "role_type"]]
        .sort_values(["watchitem_id", "role_type", "person_id"])
        .reset_index(drop=True)
    )

    # -----------------------------------------------------
    # 3) Coleção para recommendation query (Q4)
    # -----------------------------------------------------
    watchitems_by_genre_type = (
        watchitems.merge(genres, on="genre_id", how="left")
        [["genre_id", "name", "item_type", "watchitem_id", "title", "release_year", "avg_rating"]]
        .rename(columns={"name": "genre_name"})
        .sort_values(["genre_id", "item_type", "release_year"])
        .reset_index(drop=True)
    )

    return PhysicalLoadBundle(
        fragment_name="ContentFragment",
        db_model="document",
        db_engine="MongoDB",
        physical_tables={
            "persons": persons,
            "genres": genres,
            "watchitems": watchitems,
            "movies": movies,
            "series": series,
            "episodes": episodes,
            "roles": roles,
            "rates": rates,
            "watchitems_by_title": watchitems_by_title,
            "persons_by_watchitem": persons_by_watchitem,
            "watchitems_by_genre_type": watchitems_by_genre_type,
        }
    )

In [21]:
# =========================================================
# BLOCO 7 — CONSTRUIR PAYLOADS FÍSICOS
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

# Fragmento de usuário continua no PostgreSQL
user_payload_mongo_alt = build_user_fragment_postgres_payload(
    fragmented_data["UserFragment"]
)

# Fragmento de conteúdo agora vai para MongoDB
content_payload_mongo = build_content_fragment_mongodb_payload(
    fragmented_data["ContentFragment"]
)

print("Resumo do payload PostgreSQL:")
display(user_payload_mongo_alt.summary())

print("Resumo do payload MongoDB:")
display(content_payload_mongo.summary())

print("Prévia: users (PostgreSQL)")
display(user_payload_mongo_alt.physical_tables["users"].head())

print("Prévia: watchitems_by_title (MongoDB)")
display(content_payload_mongo.physical_tables["watchitems_by_title"].head())

print("Prévia: persons_by_watchitem (MongoDB)")
display(content_payload_mongo.physical_tables["persons_by_watchitem"].head())

print("Prévia: watchitems_by_genre_type (MongoDB)")
display(content_payload_mongo.physical_tables["watchitems_by_genre_type"].head())

Resumo do payload PostgreSQL:


,physical_table,rows,columns
0,users,100,"[user_id, username, password, email, last_login]"


Resumo do payload MongoDB:


,physical_table,rows,columns
0,episodes,31,"[watchitem_id, season, episode_number, length_..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[watchitem_id, length_min, media, income]"
3,persons,120,"[person_id, name, date_of_birth, gender]"
4,persons_by_watchitem,290,"[watchitem_id, person_id, name, role_type]"
5,rates,547,"[rate_id, user_id, watchitem_id, rating, verba..."
6,roles,290,"[role_id, person_id, watchitem_id, role_type]"
7,series,15,"[watchitem_id, seasons, network]"
8,watchitems,80,"[watchitem_id, title, release_year, avg_rating..."
9,watchitems_by_genre_type,80,"[genre_id, genre_name, item_type, watchitem_id..."


Prévia: users (PostgreSQL)


...

,watchitem_id,title,release_year,avg_rating,genre_id,item_type
0,1,Title 1,2013,4.66,10,Episode
1,2,Title 2,2019,2.50,5,Movie
2,3,Title 3,1990,1.81,4,Episode
3,4,Title 4,2016,5.16,8,Episode
4,5,Title 5,1995,5.51,8,Movie


Prévia: persons_by_watchitem (MongoDB)


,watchitem_id,person_id,name,role_type
0,1,58,Person 58,Actor
1,1,29,Person 29,Director
2,1,54,Person 54,Producer
3,2,21,Person 21,Director
4,2,104,Person 104,Director


Prévia: watchitems_by_genre_type (MongoDB)


,genre_id,genre_name,item_type,watchitem_id,title,release_year,avg_rating
0,1,Action,Episode,19,Title 19,2017,7.48
1,1,Action,Episode,53,Title 53,2017,6.95
2,1,Action,Episode,56,Title 56,2023,3.17
3,1,Action,Movie,15,Title 15,1993,9.12
4,1,Action,Movie,60,Title 60,1997,3.79


In [22]:
# =========================================================
# BLOCO A1.1 — UTILITÁRIOS DE EXPORTAÇÃO
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json


# ---------------------------------------------------------
# 1) Criar diretório se não existir
# ---------------------------------------------------------
def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


# ---------------------------------------------------------
# 2) Escapar strings para SQL
# ---------------------------------------------------------
def escape_sql_string(value: str) -> str:
    """
    Troca ' por '' para evitar quebrar INSERTs SQL.
    """
    return value.replace("'", "''")


# ---------------------------------------------------------
# 3) Converter valor Python/Pandas para literal SQL
# ---------------------------------------------------------
def to_sql_literal(value):
    """
    Exemplos:
    - 10        -> 10
    - 3.14      -> 3.14
    - "abc"     -> 'abc'
    - None/NaN  -> NULL
    """
    if pd.isna(value):
        return "NULL"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return str(float(value))
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    return f"'{escape_sql_string(str(value))}'"


# ---------------------------------------------------------
# 4) Inferir tipo PostgreSQL simples a partir da coluna
# ---------------------------------------------------------
def infer_postgres_type(series: pd.Series) -> str:
    """
    Inferência simples e pragmática.
    """
    if pd.api.types.is_integer_dtype(series):
        return "BIGINT"
    if pd.api.types.is_float_dtype(series):
        return "DOUBLE PRECISION"
    if pd.api.types.is_bool_dtype(series):
        return "BOOLEAN"

    return "TEXT"


# ---------------------------------------------------------
# 5) Normalizar DataFrame para documentos MongoDB/JSON
# ---------------------------------------------------------
def dataframe_to_mongo_documents(df: pd.DataFrame) -> list[dict]:
    """
    Converte um DataFrame em lista de documentos Python puros,
    substituindo NaN por None e convertendo tipos numpy para tipos nativos.
    """

    def normalize_value(v):
        if pd.isna(v):
            return None
        if isinstance(v, np.integer):
            return int(v)
        if isinstance(v, np.floating):
            return float(v)
        if isinstance(v, np.bool_):
            return bool(v)
        return v

    records = df.to_dict(orient="records")
    normalized_records = []

    for record in records:
        normalized_records.append({
            k: normalize_value(v)
            for k, v in record.items()
        })

    return normalized_records


# ---------------------------------------------------------
# 6) Salvar texto em arquivo
# ---------------------------------------------------------
def write_text_file(path: str | Path, content: str) -> Path:
    path = Path(path)
    path.write_text(content, encoding="utf-8")
    return path

In [23]:
# =========================================================
# BLOCO A1.2 — EXPORTAR TABELAS FÍSICAS PARA CSV E JSON
# =========================================================

def export_bundle_to_csv_json(
    payload: PhysicalLoadBundle,
    base_output_dir: str | Path
) -> dict[str, list[Path]]:
    """
    Exporta cada tabela física do payload para:
    - CSV
    - JSON (records)

    Estrutura:
    base_output_dir/
        fragment_name/
            csv/
            json/
    """

    base_output_dir = ensure_dir(base_output_dir)
    fragment_dir = ensure_dir(base_output_dir / payload.fragment_name)
    csv_dir = ensure_dir(fragment_dir / "csv")
    json_dir = ensure_dir(fragment_dir / "json")

    exported_csv = []
    exported_json = []

    for table_name, df in payload.physical_tables.items():
        csv_path = csv_dir / f"{table_name}.csv"
        json_path = json_dir / f"{table_name}.json"

        df.to_csv(csv_path, index=False)
        json_path.write_text(
        json.dumps(dataframe_to_mongo_documents(df), indent=2, ensure_ascii=False),
        encoding="utf-8"
)
        exported_csv.append(csv_path)
        exported_json.append(json_path)

    return {
        "csv": exported_csv,
        "json": exported_json,
    }

In [24]:
# =========================================================
# BLOCO A1.3 — GERAR SCRIPT SQL PARA POSTGRESQL
# =========================================================

def build_postgres_create_table_sql(
    table_name: str,
    df: pd.DataFrame,
    primary_key: str | None = None
) -> str:
    """
    Gera um CREATE TABLE simples para PostgreSQL.
    """
    column_defs = []

    for col in df.columns:
        pg_type = infer_postgres_type(df[col])

        if primary_key is not None and col == primary_key:
            column_defs.append(f'    "{col}" {pg_type} PRIMARY KEY')
        else:
            column_defs.append(f'    "{col}" {pg_type}')

    sql = f'CREATE TABLE IF NOT EXISTS "{table_name}" (\n'
    sql += ",\n".join(column_defs)
    sql += "\n);\n"

    return sql


def build_postgres_insert_sql(table_name: str, df: pd.DataFrame) -> str:
    """
    Gera INSERTs simples para PostgreSQL.
    """
    columns_sql = ", ".join([f'"{col}"' for col in df.columns])

    lines = []
    for _, row in df.iterrows():
        values_sql = ", ".join([to_sql_literal(row[col]) for col in df.columns])
        lines.append(f'INSERT INTO "{table_name}" ({columns_sql}) VALUES ({values_sql});')

    return "\n".join(lines) + "\n"


def export_postgres_payload_sql(
    payload: PhysicalLoadBundle,
    output_dir: str | Path,
    primary_keys: dict[str, str] | None = None
) -> Path:
    """
    Exporta um único arquivo .sql contendo CREATE TABLE + INSERT.
    """
    if primary_keys is None:
        primary_keys = {}

    output_dir = ensure_dir(output_dir)
    sql_parts = []

    for table_name, df in payload.physical_tables.items():
        pk = primary_keys.get(table_name)

        sql_parts.append(f"-- ==================================================")
        sql_parts.append(f"-- TABLE: {table_name}")
        sql_parts.append(f"-- ==================================================\n")

        sql_parts.append(build_postgres_create_table_sql(table_name, df, primary_key=pk))
        sql_parts.append(build_postgres_insert_sql(table_name, df))
        sql_parts.append("\n")

    final_sql = "\n".join(sql_parts)

    out_path = output_dir / f"{payload.fragment_name}_postgres.sql"
    write_text_file(out_path, final_sql)
    return out_path

In [25]:
# =========================================================
# BLOCO A1.5 — EXECUTAR EXPORTAÇÃO COMPLETA
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

# ---------------------------------------------------------
# 1) Checagens simples
# ---------------------------------------------------------
required_names = [
    "user_payload_mongo_alt",
    "content_payload_mongo",
    "export_bundle_to_csv_json",
    "export_postgres_payload_sql",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Diretório-base de saída
# ---------------------------------------------------------
EXPORT_BASE_DIR = ensure_dir("exports/imdb_hubara_mongo_manual")

# ---------------------------------------------------------
# 3) Exportar CSV/JSON
# ---------------------------------------------------------
user_files = export_bundle_to_csv_json(user_payload_mongo_alt, EXPORT_BASE_DIR)
content_files = export_bundle_to_csv_json(content_payload_mongo, EXPORT_BASE_DIR)

print("Arquivos CSV/JSON exportados para UserFragment:")
for kind, paths in user_files.items():
    print(f"\n{kind.upper()}:")
    for p in paths:
        print(p)

print("\nArquivos CSV/JSON exportados para ContentFragment:")
for kind, paths in content_files.items():
    print(f"\n{kind.upper()}:")
    for p in paths:
        print(p)

# ---------------------------------------------------------
# 4) Exportar SQL PostgreSQL
# ---------------------------------------------------------
postgres_primary_keys = {
    "users": "user_id",
}

postgres_sql_path = export_postgres_payload_sql(
    payload=user_payload_mongo_alt,
    output_dir=EXPORT_BASE_DIR / "sql",
    primary_keys=postgres_primary_keys
)

print("\nScript SQL PostgreSQL gerado:")
print(postgres_sql_path)

# ---------------------------------------------------------
# 5) Guardar caminho base do conteúdo Mongo
# ---------------------------------------------------------
mongo_json_dir = EXPORT_BASE_DIR / "ContentFragment" / "json"

print("\nDiretório JSON do ContentFragment (MongoDB):")
print(mongo_json_dir)

Arquivos CSV/JSON exportados para UserFragment:

CSV:
exports/imdb_hubara_mongo_manual/UserFragment/csv/users.csv

JSON:
exports/imdb_hubara_mongo_manual/UserFragment/json/users.json

Arquivos CSV/JSON exportados para ContentFragment:

CSV:
exports/imdb_hubara_mongo_manual/ContentFragment/csv/persons.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/genres.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/watchitems.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/movies.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/series.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/episodes.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/roles.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/rates.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/watchitems_by_title.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/persons_by_watchitem.csv
exports/imdb_hubara_mongo_manual/ContentFragment/csv/watchitems_by_genre_type.csv

JSON

In [26]:
# =========================================================
# BLOCO A1.6 — INSPECIONAR OS ARTEFATOS GERADOS
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

from pathlib import Path
import json

# ---------------------------------------------------------
# 1) Prévia do script PostgreSQL
# ---------------------------------------------------------
print("Prévia do script PostgreSQL:\n")
print(Path(postgres_sql_path).read_text(encoding="utf-8")[:3000])

print("\n" + "=" * 80 + "\n")

# ---------------------------------------------------------
# 2) Prévia de um JSON do ContentFragment (MongoDB)
# ---------------------------------------------------------
sample_json_path = mongo_json_dir / "watchitems_by_title.json"

print(f"Prévia do JSON MongoDB: {sample_json_path}\n")

json_text = Path(sample_json_path).read_text(encoding="utf-8")
print(json_text[:3000])

Prévia do script PostgreSQL:

-- ==================================================
-- TABLE: users
-- ==================================================

CREATE TABLE IF NOT EXISTS "users" (
    "user_id" BIGINT PRIMARY KEY,
    "username" TEXT,
    "password" TEXT,
    "email" TEXT,
    "last_login" TEXT
);

INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (1, 'user_1', 'pass_1', 'user_1@mail.com', '2026-01-02');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (2, 'user_2', 'pass_2', 'user_2@mail.com', '2026-01-03');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (3, 'user_3', 'pass_3', 'user_3@mail.com', '2026-01-04');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (4, 'user_4', 'pass_4', 'user_4@mail.com', '2026-01-05');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (5, 'user_5', 'pass_5', 'u

In [27]:
pip install psycopg[binary] cassandra-driver pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 54.0 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [37]:
# =========================================================
# BLOCO A2.1 — IMPORTS E FUNÇÕES UTILITÁRIAS
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

from pathlib import Path
import re

# PostgreSQL
try:
    import psycopg
    HAS_PSYCOPG3 = True
except ImportError:
    HAS_PSYCOPG3 = False
    psycopg = None

# MongoDB
try:
    from pymongo import MongoClient
    HAS_PYMONGO = True
except ImportError:
    HAS_PYMONGO = False
    MongoClient = None


def safe_identifier(name: str) -> str:
    """
    Garante que nomes como database_name sejam simples e seguros.
    Aceita apenas letras, números e underscore, e não pode começar com número.
    """
    if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", name):
        raise ValueError(f"Identificador inválido: {name}")
    return name


def strip_comment_lines(script_text: str) -> str:
    """
    Remove linhas que começam com '--'.
    Isso ajuda a executar scripts SQL statement por statement.
    """
    cleaned_lines = []
    for line in script_text.splitlines():
        stripped = line.strip()
        if stripped.startswith("--"):
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)


def split_script_statements(script_text: str) -> list[str]:
    """
    Divide um script em statements usando ';' como separador.
    Como nossos scripts são simples, isso é suficiente.
    """
    script_text = strip_comment_lines(script_text)
    parts = script_text.split(";")
    statements = []

    for part in parts:
        stmt = part.strip()
        if stmt:
            statements.append(stmt + ";")

    return statements

In [38]:
# =========================================================
# BLOCO A2.2 — POSTGRESQL: CRIAR DATABASE E EXECUTAR SCRIPT
# =========================================================

def pg_connect(dbname: str, spec: PhysicalDBSpec):
    """
    Abre conexão com PostgreSQL usando psycopg3.
    """
    if not HAS_PSYCOPG3:
        raise ImportError(
            "psycopg não está instalado. Rode: pip install psycopg[binary]"
        )

    conn = psycopg.connect(
        host=spec.host,
        port=spec.port,
        dbname=dbname,
        user=spec.username,
        password=spec.password,
        autocommit=True
    )
    return conn


def ensure_postgres_database_exists(spec: PhysicalDBSpec):
    """
    Cria o database alvo se ele ainda não existir.
    Conecta primeiro no database padrão 'postgres'.
    """
    target_db = safe_identifier(spec.database_name)

    with pg_connect("postgres", spec) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1 FROM pg_database WHERE datname = %s;", (target_db,))
            exists = cur.fetchone() is not None

            if not exists:
                cur.execute(f'CREATE DATABASE "{target_db}";')
                print(f"Database PostgreSQL criado: {target_db}")
            else:
                print(f"Database PostgreSQL já existe: {target_db}")


def execute_postgres_sql_script(spec: PhysicalDBSpec, sql_script_path: str | Path):
    """
    Executa um arquivo .sql statement por statement no database alvo.
    """
    sql_script_path = Path(sql_script_path)
    target_db = safe_identifier(spec.database_name)

    script_text = sql_script_path.read_text(encoding="utf-8")
    statements = split_script_statements(script_text)

    with pg_connect(target_db, spec) as conn:
        with conn.cursor() as cur:
            for stmt in statements:
                cur.execute(stmt)

    print(f"Script SQL executado com sucesso em PostgreSQL: {sql_script_path}")


def verify_postgres_table_counts(spec: PhysicalDBSpec, table_names: list[str]) -> pd.DataFrame:
    """
    Verifica quantidade de linhas por tabela no PostgreSQL.
    """
    rows = []
    target_db = safe_identifier(spec.database_name)

    with pg_connect(target_db, spec) as conn:
        with conn.cursor() as cur:
            for table_name in table_names:
                safe_table = safe_identifier(table_name)
                cur.execute(f'SELECT COUNT(*) FROM "{safe_table}";')
                count = cur.fetchone()[0]
                rows.append({
                    "table_name": table_name,
                    "row_count": count
                })

    return pd.DataFrame(rows)

In [42]:
# =========================================================
# BLOCO A2.3 — MONGODB: CONECTAR, CARREGAR E VALIDAR
# =========================================================

def mongo_connect(spec: PhysicalDBSpec):
    """
    Abre conexão com MongoDB e retorna:
    - client
    - db

    Compatível tanto com PhysicalDBSpec antigo quanto com a versão
    que possui connection_uri.
    """
    if not HAS_PYMONGO:
        raise ImportError(
            "pymongo não está instalado. Rode: pip install pymongo"
        )

    connection_uri = getattr(spec, "connection_uri", None)

    if connection_uri:
        client = MongoClient(
            connection_uri,
            serverSelectionTimeoutMS=5000
        )
    else:
        client = MongoClient(
            host=spec.host,
            port=spec.port,
            username=spec.username,
            password=spec.password,
            authSource="admin",
            serverSelectionTimeoutMS=5000
        )

    # força teste de conexão
    client.admin.command("ping")

    db = client[spec.database_name]
    return client, db

def ensure_mongodb_database_exists(spec: PhysicalDBSpec):
    """
    No MongoDB, o database passa a existir de fato quando uma coleção é criada
    ou quando um documento é inserido. Aqui apenas testamos a conexão e
    retornamos o handle do database.
    """
    client, db = mongo_connect(spec)
    try:
        _ = db.name
        print(f"Database MongoDB acessível: {db.name}")
    finally:
        client.close()


def load_mongodb_payload(
    spec: PhysicalDBSpec,
    payload: PhysicalLoadBundle,
    drop_existing: bool = True
):
    """
    Carrega o payload nas coleções MongoDB.

    Estratégia:
    - cada tabela física vira uma coleção;
    - cada linha do DataFrame vira um documento;
    - opcionalmente apaga a coleção antes de carregar.
    """
    client, db = mongo_connect(spec)

    try:
        for collection_name, df in payload.physical_tables.items():
            collection = db[collection_name]

            if drop_existing:
                collection.drop()

            docs = dataframe_to_mongo_documents(df)

            if docs:
                collection.insert_many(docs)

        print(f"Payload carregado com sucesso em MongoDB: {spec.database_name}")
    finally:
        client.close()


def create_mongodb_indexes(spec: PhysicalDBSpec):
    """
    Cria índices básicos para apoiar o workload.
    """
    client, db = mongo_connect(spec)

    try:
        # Q2: busca por título
        db["watchitems_by_title"].create_index([("title", 1)])

        # Q5: pessoas por watchitem e role_type
        db["persons_by_watchitem"].create_index([("watchitem_id", 1), ("role_type", 1)])

        # Q4: recommendation query
        db["watchitems_by_genre_type"].create_index(
            [("genre_id", 1), ("item_type", 1), ("release_year", 1)]
        )

        # índices úteis adicionais
        db["persons"].create_index([("person_id", 1)], unique=True)
        db["genres"].create_index([("genre_id", 1)], unique=True)
        db["watchitems"].create_index([("watchitem_id", 1)], unique=True)
        db["movies"].create_index([("watchitem_id", 1)])
        db["series"].create_index([("watchitem_id", 1)])
        db["episodes"].create_index([("watchitem_id", 1)])
        db["roles"].create_index([("role_id", 1)], unique=True)
        db["rates"].create_index([("rate_id", 1)], unique=True)

        print(f"Índices MongoDB criados com sucesso em: {spec.database_name}")
    finally:
        client.close()


def verify_mongodb_collection_counts(spec: PhysicalDBSpec, collection_names: list[str]) -> pd.DataFrame:
    """
    Verifica quantidade de documentos por coleção no MongoDB.
    """
    client, db = mongo_connect(spec)
    rows = []

    try:
        for collection_name in collection_names:
            count = db[collection_name].count_documents({})
            rows.append({
                "collection_name": collection_name,
                "row_count": count
            })
    finally:
        client.close()

    return pd.DataFrame(rows)

In [46]:
#para conectar no servidor remoto - rodar no terminal 
#ssh -N \
#  -L 55432:127.0.0.1:5432 \
#  -L 59042:127.0.0.1:9042 \
#  -L 57017:127.0.0.1:27017 \
#  hudson@150.162.57.138

PostgreSQL: falha na conexão
ConnectionTimeout connection timeout expired
Cassandra: falha na conexão
NoHostAvailable ('Unable to connect to any servers', {'150.162.57.138:9042': OSError(None, "Tried connecting to [('150.162.57.138', 9042)]. Last error: timed out")})


In [40]:
# =========================================================
# TESTE DE CONECTIVIDADE VIA SSH TUNNEL
# =========================================================

TUNNEL_HOST = "127.0.0.1"

# PostgreSQL
try:
    import psycopg
    conn = psycopg.connect(
        host=TUNNEL_HOST,
        port=55432,
        dbname="postgres",
        user="postgres",
        password="postgres",
        connect_timeout=5
    )
    conn.close()
    print("PostgreSQL via túnel: conexão OK")
except Exception as e:
    print("PostgreSQL via túnel: falha na conexão")
    print(type(e).__name__, e)

# Cassandra
try:
    from cassandra.cluster import Cluster
    cluster = Cluster(contact_points=[TUNNEL_HOST], port=59042)
    session = cluster.connect()
    session.shutdown()
    cluster.shutdown()
    print("Cassandra via túnel: conexão OK")
except Exception as e:
    print("Cassandra via túnel: falha na conexão")
    print(type(e).__name__, e)


#mongoDB

# =========================================================
# TESTE DE CONECTIVIDADE VIA SSH TUNNEL — MONGODB
# =========================================================

from pymongo import MongoClient

try:
    client = MongoClient(
        host="127.0.0.1",
        port=57017,
        username="mongo",
        password="mongo",
        authSource="admin",
        serverSelectionTimeoutMS=5000
    )
    client.admin.command("ping")
    print("MongoDB via túnel: conexão OK")
    client.close()
except Exception as e:
    print("MongoDB via túnel: falha na conexão")
    print(type(e).__name__, e)

PostgreSQL via túnel: conexão OK
Cassandra via túnel: conexão OK
MongoDB via túnel: conexão OK


In [43]:
# =========================================================
# BLOCO A2.4 — EXECUTAR CARGA REAL NOS BANCOS
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

required_names = [
    "hubara_imdb_mongo_materialization",
    "postgres_sql_path",
    "user_payload_mongo_alt",
    "content_payload_mongo",
    "ensure_postgres_database_exists",
    "execute_postgres_sql_script",
    "ensure_mongodb_database_exists",
    "load_mongodb_payload",
    "create_mongodb_indexes",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 1) Recuperar specs físicos
# ---------------------------------------------------------
user_db_spec = hubara_imdb_mongo_materialization.fragment_to_db["UserFragment"]
content_db_spec = hubara_imdb_mongo_materialization.fragment_to_db["ContentFragment"]

print("User DB Spec:", user_db_spec)
print("Content DB Spec:", content_db_spec)

# ---------------------------------------------------------
# 2) Garantir database PostgreSQL
# ---------------------------------------------------------
ensure_postgres_database_exists(user_db_spec)

# ---------------------------------------------------------
# 3) Garantir acesso ao MongoDB
# ---------------------------------------------------------
ensure_mongodb_database_exists(content_db_spec)

# ---------------------------------------------------------
# 4) Executar script SQL no PostgreSQL
# ---------------------------------------------------------
execute_postgres_sql_script(user_db_spec, postgres_sql_path)

# ---------------------------------------------------------
# 5) Carregar payload no MongoDB
# ---------------------------------------------------------
load_mongodb_payload(
    spec=content_db_spec,
    payload=content_payload_mongo,
    drop_existing=True
)

# ---------------------------------------------------------
# 6) Criar índices MongoDB
# ---------------------------------------------------------
create_mongodb_indexes(content_db_spec)

print("\nCarga real finalizada.")

User DB Spec: PhysicalDBSpec(model='relational', engine='PostgreSQL', host='127.0.0.1', port=55432, database_name='imdb_user_db_mongo_alt', username='postgres', password='postgres')
Content DB Spec: PhysicalDBSpec(model='document', engine='MongoDB', host='127.0.0.1', port=57017, database_name='imdb_content_mongo_db', username='mongo', password='mongo')
Database PostgreSQL já existe: imdb_user_db_mongo_alt
Database MongoDB acessível: imdb_content_mongo_db
Script SQL executado com sucesso em PostgreSQL: exports/imdb_hubara_mongo_manual/sql/UserFragment_postgres.sql
Payload carregado com sucesso em MongoDB: imdb_content_mongo_db
Índices MongoDB criados com sucesso em: imdb_content_mongo_db

Carga real finalizada.


In [44]:
# =========================================================
# BLOCO A2.5 — VALIDAR CONTAGEM DAS TABELAS/COLEÇÕES
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

# ---------------------------------------------------------
# 1) PostgreSQL
# ---------------------------------------------------------
postgres_tables = list(user_payload_mongo_alt.physical_tables.keys())
postgres_counts_df = verify_postgres_table_counts(user_db_spec, postgres_tables)

print("Contagens no PostgreSQL:")
display(postgres_counts_df)

# ---------------------------------------------------------
# 2) MongoDB
# ---------------------------------------------------------
mongo_collections = list(content_payload_mongo.physical_tables.keys())
mongo_counts_df = verify_mongodb_collection_counts(content_db_spec, mongo_collections)

print("Contagens no MongoDB:")
display(mongo_counts_df)

Contagens no PostgreSQL:


,table_name,row_count
0,users,100


Contagens no MongoDB:


,collection_name,row_count
0,persons,120
1,genres,10
2,watchitems,80
3,movies,34
4,series,15
5,episodes,31
6,roles,290
7,rates,547
8,watchitems_by_title,80
9,persons_by_watchitem,290


In [45]:
# =========================================================
# BLOCO B1 — FUNÇÕES DE BENCHMARK E ESTATÍSTICAS MELHORADAS
# =========================================================

import time
import random
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# 1) Gerar parâmetros válidos para cada query
# ---------------------------------------------------------
def build_benchmark_parameter_pool(user_payload, content_payload, seed: int = 42):
    users_df = user_payload.physical_tables["users"]
    watchitems_df = content_payload.physical_tables["watchitems"]
    persons_by_watchitem_df = content_payload.physical_tables["persons_by_watchitem"]
    watchitems_by_genre_type_df = content_payload.physical_tables["watchitems_by_genre_type"]

    # -----------------------------
    # Q4: normalizar tipos
    # -----------------------------
    q4_df = watchitems_by_genre_type_df[
        ["genre_id", "item_type", "release_year"]
    ].drop_duplicates().copy()

    q4_df["genre_id"] = q4_df["genre_id"].astype(int)
    q4_df["item_type"] = q4_df["item_type"].astype(str)
    q4_df["release_year"] = q4_df["release_year"].astype(int)

    # -----------------------------
    # Q5: normalizar tipos
    # -----------------------------
    q5_df = persons_by_watchitem_df[
        ["watchitem_id", "role_type"]
    ].drop_duplicates().copy()

    q5_df["watchitem_id"] = q5_df["watchitem_id"].astype(int)
    q5_df["role_type"] = q5_df["role_type"].astype(str)

    pool = {
        "Q1_Login": users_df["username"].astype(str).tolist(),
        "Q2_SimpleSearch": watchitems_df["title"].astype(str).tolist(),
        "Q4_Recommendation": q4_df.to_dict(orient="records"),
        "Q5_AllPersonsOfTypeForWatchItem": q5_df.to_dict(orient="records"),
    }

    return pool


# ---------------------------------------------------------
# 2) Média sem extremos
# ---------------------------------------------------------
def trimmed_mean(values, trim_each_side: int = 1) -> float:
    """
    Remove os menores e maiores tempos antes de calcular a média.

    Exemplo:
    valores = [10, 11, 12, 13, 100]
    trim_each_side = 1
    remove 10 e 100 -> média de [11, 12, 13]
    """
    arr = np.array(values, dtype=float)

    if len(arr) == 0:
        return np.nan

    arr = np.sort(arr)

    # Se não houver dados suficientes para remover extremos,
    # cai para média normal.
    if len(arr) <= 2 * trim_each_side:
        return float(np.mean(arr))

    trimmed = arr[trim_each_side: len(arr) - trim_each_side]
    return float(np.mean(trimmed))


# ---------------------------------------------------------
# 3) Funções auxiliares para percentis
# ---------------------------------------------------------
def p95(x):
    return float(np.percentile(x, 95))

def p99(x):
    return float(np.percentile(x, 99))


# ---------------------------------------------------------
# 4) Resumo do benchmark com estatísticas extras
# ---------------------------------------------------------
def summarize_benchmark_results(
    result_df: pd.DataFrame,
    trim_each_side: int = 1,
    separate_by_phase: bool = True
) -> pd.DataFrame:
    """
    Gera estatísticas agregadas por query.
    Se separate_by_phase=True, separa cold e hot.
    """

    if result_df.empty:
        return pd.DataFrame()

    ok_df = result_df[result_df["success"] == True].copy()

    group_cols = ["query_name", "fragment_name", "db_engine"]
    if separate_by_phase:
        group_cols.append("benchmark_phase")

    summary = (
        ok_df.groupby(group_cols, as_index=False)
        .agg(
            runs=("latency_ms", "count"),
            avg_ms=("latency_ms", "mean"),
            median_ms=("latency_ms", "median"),
            min_ms=("latency_ms", "min"),
            max_ms=("latency_ms", "max"),
            std_ms=("latency_ms", "std"),
            var_ms=("latency_ms", "var"),
            p95_ms=("latency_ms", p95),
            p99_ms=("latency_ms", p99),
        )
    )

    summary["std_ms"] = summary["std_ms"].fillna(0.0)
    summary["var_ms"] = summary["var_ms"].fillna(0.0)

    summary["range_ms"] = summary["max_ms"] - summary["min_ms"]

    summary["cv_pct"] = np.where(
        summary["avg_ms"] > 0,
        (summary["std_ms"] / summary["avg_ms"]) * 100.0,
        np.nan
    )

    trimmed_rows = []
    for keys, group in ok_df.groupby(group_cols):
        latencies = group["latency_ms"].tolist()
        trimmed_avg = trimmed_mean(latencies, trim_each_side=trim_each_side)

        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))
        row["avg_trimmed_ms"] = trimmed_avg
        trimmed_rows.append(row)

    trimmed_df = pd.DataFrame(trimmed_rows)

    summary = summary.merge(trimmed_df, on=group_cols, how="left")

    summary = summary.sort_values(group_cols).reset_index(drop=True)

    return summary

In [46]:
# =========================================================
# BLOCO B2 — ABRIR CONEXÕES DE BENCHMARK
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

# Reusa funções já criadas antes:
# - pg_connect(...)
# - mongo_connect(...)

def open_benchmark_connections(user_db_spec, content_db_spec):
    """
    Abre conexões persistentes para o benchmark.
    """
    # PostgreSQL
    pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
    pg_conn.autocommit = True

    # MongoDB
    mongo_client, mongo_db = mongo_connect(content_db_spec)

    return pg_conn, mongo_client, mongo_db


def close_benchmark_connections(pg_conn, mongo_client, mongo_db):
    """
    Fecha tudo de forma limpa.
    """
    try:
        if pg_conn is not None:
            pg_conn.close()
    except Exception:
        pass

    try:
        if mongo_client is not None:
            mongo_client.close()
    except Exception:
        pass

In [47]:
# =========================================================
# BLOCO B3 — EXECUÇÃO REAL DAS QUERIES DO WORKLOAD
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

# ---------------------------------------------------------
# Q1 — Login (PostgreSQL)
# RETURN password FROM User WHERE username = ?
# ---------------------------------------------------------
def run_q1_login_postgres(pg_conn, username: str):
    sql = """
    SELECT password
    FROM users
    WHERE username = %s
    LIMIT 1;
    """
    with pg_conn.cursor() as cur:
        cur.execute(sql, (username,))
        return cur.fetchone()


# ---------------------------------------------------------
# Q2 — Simple search (MongoDB)
# RETURN ALL FROM WatchItem WHERE title = ?
# Usa a coleção orientada ao workload: watchitems_by_title
# ---------------------------------------------------------
def run_q2_simple_search_mongodb(mongo_db, title: str):
    cursor = mongo_db["watchitems_by_title"].find(
        {"title": str(title)},
        {"_id": 0}
    )
    return list(cursor)


# ---------------------------------------------------------
# Q3 — Add entities and relationships (MongoDB)
# Inserimos:
# - uma nova pessoa em persons
# - um novo vínculo em roles
# - uma nova entrada em persons_by_watchitem
#
# Isso mantém coerência com a Q5, que consulta persons_by_watchitem.
# ---------------------------------------------------------
def run_q3_add_entities_and_relationships_mongodb(
    mongo_db,
    person_id: int,
    person_name: str,
    birth_date: str,
    gender: str,
    role_id: int,
    watchitem_id: int,
    role_type: str
):
    person_doc = {
        "person_id": int(person_id),
        "name": str(person_name),
        "date_of_birth": str(birth_date),
        "gender": str(gender)
    }

    role_doc = {
        "role_id": int(role_id),
        "person_id": int(person_id),
        "watchitem_id": int(watchitem_id),
        "role_type": str(role_type)
    }

    persons_by_watchitem_doc = {
        "watchitem_id": int(watchitem_id),
        "person_id": int(person_id),
        "name": str(person_name),
        "role_type": str(role_type)
    }

    mongo_db["persons"].insert_one(person_doc)
    mongo_db["roles"].insert_one(role_doc)
    mongo_db["persons_by_watchitem"].insert_one(persons_by_watchitem_doc)

    return True


# ---------------------------------------------------------
# Q4 — Recommendation query (MongoDB)
# Usa a coleção orientada ao workload: watchitems_by_genre_type
#
# Filtro:
# - genre_id
# - item_type
# - release_year entre year_low e year_high
# ---------------------------------------------------------
def run_q4_recommendation_mongodb(
    mongo_db,
    genre_id: int,
    item_type: str,
    year_low: int,
    year_high: int
):
    query = {
        "genre_id": int(genre_id),
        "item_type": str(item_type),
        "release_year": {
            "$gte": int(year_low),
            "$lte": int(year_high)
        }
    }

    cursor = mongo_db["watchitems_by_genre_type"].find(query, {"_id": 0})
    return list(cursor)


# ---------------------------------------------------------
# Q5 — All persons of type for a watch item (MongoDB)
# persons_by_watchitem WHERE watchitem_id = ? AND role_type = ?
# ---------------------------------------------------------
def run_q5_all_persons_for_watchitem_mongodb(
    mongo_db,
    watchitem_id: int,
    role_type: str
):
    query = {
        "watchitem_id": int(watchitem_id),
        "role_type": str(role_type)
    }

    cursor = mongo_db["persons_by_watchitem"].find(query, {"_id": 0})
    return list(cursor)

In [48]:
# =========================================================
# BLOCO B4 — EXECUTOR COMPLETO DO BENCHMARK COM COLD/HOT
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

def run_hubara_imdb_benchmark_mongo(
    user_db_spec,
    content_db_spec,
    user_payload,
    content_payload,
    cold_repetitions: int = 3,
    hot_repetitions: int = 20,
    seed: int = 42
) -> BenchmarkResult:

    rng = random.Random(seed)
    parameter_pool = build_benchmark_parameter_pool(
        user_payload=user_payload,
        content_payload=content_payload,
        seed=seed
    )

    max_person_id = int(content_payload.physical_tables["persons"]["person_id"].max())
    max_role_id = int(content_payload.physical_tables["roles"]["role_id"].max())
    available_watchitem_ids = content_payload.physical_tables["watchitems"]["watchitem_id"].tolist()

    benchmark_result = BenchmarkResult(
        scenario_name="IMDb",
        recommendation_name="Hubara_IMDb_Recommendation_MongoAlternative"
    )

    def append_result(
        query_name,
        fragment_name,
        db_engine,
        run_id,
        benchmark_phase,
        start_time,
        success,
        error_message
    ):
        latency_ms = (time.perf_counter() - start_time) * 1000.0

        benchmark_result.query_results.append(
            QueryExecutionResult(
                query_name=query_name,
                fragment_name=fragment_name,
                db_engine=db_engine,
                run_id=run_id,
                benchmark_phase=benchmark_phase,
                latency_ms=latency_ms,
                success=success,
                error_message=error_message,
            )
        )

    # =====================================================
    # COLD RUNS
    # Regra prática:
    # - cada execução abre e fecha conexão/sessão
    # =====================================================

    # -----------------------------------------------------
    # Q1 — PostgreSQL
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        username = str(rng.choice(parameter_pool["Q1_Login"]))

        start = time.perf_counter()
        success = True
        error_message = None

        pg_conn = None
        try:
            pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
            pg_conn.autocommit = True
            _ = run_q1_login_postgres(pg_conn, username)
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            if pg_conn is not None:
                pg_conn.close()

        append_result(
            "Q1_Login", "UserFragment", "PostgreSQL",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q2 — MongoDB
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        title = str(rng.choice(parameter_pool["Q2_SimpleSearch"]))

        start = time.perf_counter()
        success = True
        error_message = None

        mongo_client, mongo_db = None, None
        try:
            mongo_client, mongo_db = mongo_connect(content_db_spec)
            _ = run_q2_simple_search_mongodb(mongo_db, title)
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(None, mongo_client, mongo_db)

        append_result(
            "Q2_SimpleSearch", "ContentFragment", "MongoDB",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q3 — MongoDB
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        person_id = max_person_id + run_id
        role_id = max_role_id + run_id
        watchitem_id = int(rng.choice(available_watchitem_ids))
        person_name = f"Benchmark Person Cold {person_id}"
        birth_date = "1990-01-01"
        gender = "M"
        role_type = str(rng.choice(["Actor", "Director", "Producer"]))

        start = time.perf_counter()
        success = True
        error_message = None

        mongo_client, mongo_db = None, None
        try:
            mongo_client, mongo_db = mongo_connect(content_db_spec)
            _ = run_q3_add_entities_and_relationships_mongodb(
                mongo_db=mongo_db,
                person_id=person_id,
                person_name=person_name,
                birth_date=birth_date,
                gender=gender,
                role_id=role_id,
                watchitem_id=watchitem_id,
                role_type=role_type
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(None, mongo_client, mongo_db)

        append_result(
            "Q3_AddEntitiesAndRelationships", "ContentFragment", "MongoDB",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q4 — MongoDB
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        genre_id = int(row["genre_id"])
        item_type = str(row["item_type"])
        center_year = int(row["release_year"])

        year_low = center_year - 10
        year_high = center_year + 10

        start = time.perf_counter()
        success = True
        error_message = None

        mongo_client, mongo_db = None, None
        try:
            mongo_client, mongo_db = mongo_connect(content_db_spec)
            _ = run_q4_recommendation_mongodb(
                mongo_db=mongo_db,
                genre_id=genre_id,
                item_type=item_type,
                year_low=year_low,
                year_high=year_high
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(None, mongo_client, mongo_db)

        append_result(
            "Q4_Recommendation", "ContentFragment", "MongoDB",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q5 — MongoDB
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        row = rng.choice(parameter_pool["Q5_AllPersonsOfTypeForWatchItem"])

        watchitem_id = int(row["watchitem_id"])
        role_type = str(row["role_type"])

        start = time.perf_counter()
        success = True
        error_message = None

        mongo_client, mongo_db = None, None
        try:
            mongo_client, mongo_db = mongo_connect(content_db_spec)
            _ = run_q5_all_persons_for_watchitem_mongodb(
                mongo_db=mongo_db,
                watchitem_id=watchitem_id,
                role_type=role_type
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(None, mongo_client, mongo_db)

        append_result(
            "Q5_AllPersonsOfTypeForWatchItem", "ContentFragment", "MongoDB",
            run_id, "cold", start, success, error_message
        )

    # =====================================================
    # HOT RUNS
    # Regra prática:
    # - reaproveita conexão/sessão abertas
    # =====================================================
    pg_conn, mongo_client, mongo_db = open_benchmark_connections(user_db_spec, content_db_spec)

    try:
        # -------------------------------------------------
        # Q1 — PostgreSQL
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            username = str(rng.choice(parameter_pool["Q1_Login"]))

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q1_login_postgres(pg_conn, username)
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q1_Login", "UserFragment", "PostgreSQL",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q2 — MongoDB
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            title = str(rng.choice(parameter_pool["Q2_SimpleSearch"]))

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q2_simple_search_mongodb(mongo_db, title)
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q2_SimpleSearch", "ContentFragment", "MongoDB",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q3 — MongoDB
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            person_id = max_person_id + cold_repetitions + run_id
            role_id = max_role_id + cold_repetitions + run_id
            watchitem_id = int(rng.choice(available_watchitem_ids))
            person_name = f"Benchmark Person Hot {person_id}"
            birth_date = "1990-01-01"
            gender = "M"
            role_type = str(rng.choice(["Actor", "Director", "Producer"]))

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q3_add_entities_and_relationships_mongodb(
                    mongo_db=mongo_db,
                    person_id=person_id,
                    person_name=person_name,
                    birth_date=birth_date,
                    gender=gender,
                    role_id=role_id,
                    watchitem_id=watchitem_id,
                    role_type=role_type
                )
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q3_AddEntitiesAndRelationships", "ContentFragment", "MongoDB",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q4 — MongoDB
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            row = rng.choice(parameter_pool["Q4_Recommendation"])

            genre_id = int(row["genre_id"])
            item_type = str(row["item_type"])
            center_year = int(row["release_year"])

            year_low = center_year - 10
            year_high = center_year + 10

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q4_recommendation_mongodb(
                    mongo_db=mongo_db,
                    genre_id=genre_id,
                    item_type=item_type,
                    year_low=year_low,
                    year_high=year_high
                )
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q4_Recommendation", "ContentFragment", "MongoDB",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q5 — MongoDB
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            row = rng.choice(parameter_pool["Q5_AllPersonsOfTypeForWatchItem"])

            watchitem_id = int(row["watchitem_id"])
            role_type = str(row["role_type"])

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q5_all_persons_for_watchitem_mongodb(
                    mongo_db=mongo_db,
                    watchitem_id=watchitem_id,
                    role_type=role_type
                )
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q5_AllPersonsOfTypeForWatchItem", "ContentFragment", "MongoDB",
                run_id, "hot", start, success, error_message
            )

    finally:
        close_benchmark_connections(pg_conn, mongo_client, mongo_db)

    return benchmark_result

In [49]:
# =========================================================
# BLOCO B5 — RODAR BENCHMARK, RESUMIR E EXPORTAR CSV
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================


#=======================================================
#VERIFICANDO SE ESTÁ TUDO OK
required_names = [
    "user_db_spec",
    "content_db_spec",
    "user_payload_mongo_alt",
    "content_payload_mongo",
    "run_hubara_imdb_benchmark_mongo",
    "summarize_benchmark_results",
]

for name in required_names:
    if name not in globals():
        raise NameError(f"'{name}' não está definido. Rode primeiro os blocos anteriores.")


#========================================================

from pathlib import Path

# ---------------------------------------------------------
# 1) Parâmetros do benchmark
# ---------------------------------------------------------
COLD_REPETITIONS = 3
HOT_REPETITIONS = 20
TRIM_EACH_SIDE = 1

# ---------------------------------------------------------
# 2) Executar benchmark
# ---------------------------------------------------------
benchmark_result = run_hubara_imdb_benchmark_mongo(
    user_db_spec=user_db_spec,
    content_db_spec=content_db_spec,
    user_payload=user_payload_mongo_alt,
    content_payload=content_payload_mongo,
    cold_repetitions=COLD_REPETITIONS,
    hot_repetitions=HOT_REPETITIONS,
    seed=42
)

# ---------------------------------------------------------
# 3) Converter resultados para DataFrame
# ---------------------------------------------------------
benchmark_df = benchmark_result.to_dataframe()

summary_df = summarize_benchmark_results(
    result_df=benchmark_df,
    trim_each_side=TRIM_EACH_SIDE,
    separate_by_phase=True
)

failures_df = benchmark_df[benchmark_df["success"] == False].copy()

# ---------------------------------------------------------
# 4) Mostrar resultados
# ---------------------------------------------------------
print("Resultados brutos do benchmark:")
display(benchmark_df.head(30))

print("Resumo estatístico do benchmark:")
display(summary_df)

print("Falhas, se existirem:")
display(failures_df)

# ---------------------------------------------------------
# 5) Exportar resultados para CSV
# ---------------------------------------------------------
results_dir = Path("exports/benchmark_results_mongo")
results_dir.mkdir(parents=True, exist_ok=True)

raw_csv_path = results_dir / "hubara_imdb_mongo_benchmark_raw.csv"
summary_csv_path = results_dir / "hubara_imdb_mongo_benchmark_summary.csv"
failures_csv_path = results_dir / "hubara_imdb_mongo_benchmark_failures.csv"

benchmark_df.to_csv(raw_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)
failures_df.to_csv(failures_csv_path, index=False)

print("\nArquivos CSV gerados:")
print(raw_csv_path)
print(summary_csv_path)
print(failures_csv_path)

Resultados brutos do benchmark:


,scenario_name,recommendation_name,query_name,fragment_name,db_engine,run_id,benchmark_phase,latency_ms,success,error_message
0,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q1_Login,UserFragment,PostgreSQL,1,cold,16.749775,True,None
1,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q1_Login,UserFragment,PostgreSQL,2,cold,13.580939,True,None
2,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q1_Login,UserFragment,PostgreSQL,3,cold,18.816519,True,None
3,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q2_SimpleSearch,ContentFragment,MongoDB,1,cold,74.049563,True,None
4,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q2_SimpleSearch,ContentFragment,MongoDB,2,cold,61.505477,True,None
5,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q2_SimpleSearch,ContentFragment,MongoDB,3,cold,64.045477,True,None
6,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q3_AddEntitiesAndRelationships,ContentFragment,MongoDB,1,cold,62.985164,True,None
7,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q3_AddEntitiesAndRelationships,ContentFragment,MongoDB,2,cold,64.288450,True,None
8,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q3_AddEntitiesAndRelationships,ContentFragment,MongoDB,3,cold,67.199103,True,None
9,IMDb,Hubara_IMDb_Recommendation_MongoAlternative,Q4_Recommendation,ContentFragment,MongoDB,1,cold,59.084338,True,None


Resumo estatístico do benchmark:


,query_name,fragment_name,db_engine,benchmark_phase,runs,avg_ms,median_ms,min_ms,max_ms,std_ms,var_ms,p95_ms,p99_ms,range_ms,cv_pct,avg_trimmed_ms
0,Q1_Login,UserFragment,PostgreSQL,cold,3,16.382411,16.749775,13.580939,18.816519,2.637052,6.954042,18.609845,18.775184,5.235580,16.096848,16.749775
1,Q1_Login,UserFragment,PostgreSQL,hot,20,1.964513,1.536232,1.271725,4.503622,0.862676,0.744210,3.498340,4.302566,3.231897,43.912983,1.861940
2,Q2_SimpleSearch,ContentFragment,MongoDB,cold,3,66.533506,64.045477,61.505477,74.049563,6.631835,43.981238,73.049155,73.849481,12.544086,9.967662,64.045477
3,Q2_SimpleSearch,ContentFragment,MongoDB,hot,20,1.746330,1.644929,0.860240,2.866703,0.508419,0.258490,2.348929,2.763148,2.006463,29.113574,1.733315
4,Q3_AddEntitiesAndRelationships,ContentFragment,MongoDB,cold,3,64.824239,64.288450,62.985164,67.199103,2.157457,4.654623,66.908038,67.140890,4.213939,3.328165,64.288450
5,Q3_AddEntitiesAndRelationships,ContentFragment,MongoDB,hot,20,3.519575,3.454875,3.111366,4.421381,0.358389,0.128443,4.177771,4.372659,1.310015,10.182738,3.492152
6,Q4_Recommendation,ContentFragment,MongoDB,cold,3,60.716737,59.166499,59.084338,63.899375,2.756551,7.598574,63.426087,63.804718,4.815037,4.540019,59.166499
7,Q4_Recommendation,ContentFragment,MongoDB,hot,20,1.508527,1.432091,1.181424,2.184116,0.265130,0.070294,1.886620,2.124617,1.002692,17.575434,1.489166
8,Q5_AllPersonsOfTypeForWatchItem,ContentFragment,MongoDB,cold,3,63.167547,63.486800,59.545175,66.470665,3.473765,12.067046,66.172278,66.410988,6.925490,5.499288,63.486800
9,Q5_AllPersonsOfTypeForWatchItem,ContentFragment,MongoDB,hot,20,1.298191,1.280655,1.050752,1.517497,0.123613,0.015280,1.486004,1.511198,0.466745,9.521930,1.299754


Falhas, se existirem:


,scenario_name,recommendation_name,query_name,fragment_name,db_engine,run_id,benchmark_phase,latency_ms,success,error_message



Arquivos CSV gerados:
exports/benchmark_results_mongo/hubara_imdb_mongo_benchmark_raw.csv
exports/benchmark_results_mongo/hubara_imdb_mongo_benchmark_summary.csv
exports/benchmark_results_mongo/hubara_imdb_mongo_benchmark_failures.csv
